# Phase B — Speech-to-speech avec tool au milieu (L4)

Étend la Phase A (audio→tool call) à la **boucle complète** :

```
audio user ─▶ [tool call en TEXTE] ─▶ exécution outil ─▶ [résultat réinjecté]
           ─▶ [réponse TEXTE+AUDIO parlée, ancrée dans le résultat]
```

« Penser en texte, parler en audio » : le tour tool-call est texte seul (aucune
frame audio gaspillée), seule la réponse finale est vocalisée.

## Latence (le point clé)
- **Un seul modèle** (audio-in + audio-out natifs) — pas de hop ASR/TTS.
- **Filler vocal** pendant l'exécution (« let me check… ») → masque le round-trip outil.
- **Stop-on-span** : on coupe la génération dès `<|tool_call_end|>` et on lance l'outil.
- **Streaming DELTA** : 1er son ~200 ms ; **prefix caching** (vLLM-Omni) en service.
- Backends outils rapides + cache ; réponses courtes.

## Données (différence Phase A)
Dialogues **multi-tours** (`--mode loop`) : on génère EN PLUS un `tool_result`
plausible et une **réponse parlée** ancrée. Le TTS vocalise **user ET assistant**
(voix assistant fixe). Packing en `--assistant-audio-mode interleaved`.


## 1. Repo + deps (vllm/vllm-omni APPARIÉS pour Voxtral, + train)

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Rcarvalo/finetuning_s2s_toolcalling"   # <-- ton repo
BRANCH   = "claude/blissful-tesla-7i1yky"
WORK     = "/content/finetuning_s2s_toolcalling"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=False)
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("vllm==0.22.0", "vllm-omni==0.22.0")            # paire APPARIÉE (Voxtral TTS)
pip("-e", f"{WORK}[train,tooldata]")
os.chdir(WORK); sys.path.insert(0, WORK + "/src"); sys.path.insert(0, WORK + "/scripts")
print("repo:", WORK)

## 2. Connexions + parametres

In [ ]:
import os, getpass
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY") or getpass.getpass("GEMINI_API_KEY: ")
from huggingface_hub import login; login()
import wandb; wandb.login()

N_TOTAL       = 3000
USER_VOICES   = ["casual_male", "casual_female", "cheerful_female"]   # voix utilisateur
ASSIST_VOICE  = "neutral_female"                                       # voix FIXE de l'assistant
TEST_VOICES   = ["neutral_male"]                                       # held-out (≠ train, ≠ assistant)
HUB_ADAPTER   = "Rcarvalo/lfm25-tc-en-s2s-adapter"
print("N_TOTAL:", N_TOTAL)

## 3. Lancer Voxtral TTS (vLLM-Omni) — cu13 + fail-fast

In [ ]:
import glob, subprocess, time, httpx
libs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/**/libcudart.so.13", recursive=True)
if not libs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu13"], check=False)
    libs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/**/libcudart.so.13", recursive=True)
env = dict(os.environ); env["LD_LIBRARY_PATH"] = ":".join(sorted({os.path.dirname(p) for p in libs}) + [env.get("LD_LIBRARY_PATH", "")])
LOG = open("/content/voxtral.log", "w")
srv = subprocess.Popen(["vllm", "serve", "mistralai/Voxtral-4B-TTS-2603", "--omni"], env=env, stdout=LOG, stderr=subprocess.STDOUT)
for _ in range(360):
    if srv.poll() is not None:
        print("SERVEUR MORT:\n", open("/content/voxtral.log").read()[-2500:]); break
    try:
        if httpx.get("http://localhost:8000/v1/models", timeout=5).status_code == 200:
            print("Voxtral pret."); break
    except Exception: pass
    time.sleep(10)

## 4. Generer les dialogues S2S (mode loop : + tool_result + reponse parlee)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/generate_toolcalling_data.py", "--mode", "loop",
    "--provider", "gemini", "--output", "data/tc_en_s2s.jsonl", "--n-total", str(N_TOTAL),
    "--held-out", "benchmark/toolcalling_en/cases.sample.jsonl"], check=True)
print(subprocess.run(["wc", "-l", "data/tc_en_s2s.jsonl"], capture_output=True, text=True).stdout)

## 5. TTS user + assistant (voix assistant FIXE ; held-out pour le test)

In [ ]:
import subprocess, sys
# split train/val/test held-out
import json, random
rows = [l for l in open("data/tc_en_s2s.jsonl")]
random.Random(0).shuffle(rows)
k = max(1, int(len(rows) * 0.05))
open("data/tc_en_s2s.val.jsonl", "w").writelines(rows[:k])
open("data/tc_en_s2s.train.jsonl", "w").writelines(rows[k:])

for split, src, sp in [("train", "data/tc_en_s2s.train.jsonl", "train"),
                       ("val",   "data/tc_en_s2s.val.jsonl",   "train")]:
    subprocess.run([sys.executable, "scripts/synthesize_user_audio.py", "--engine", "voxtral",
        "--dialogues", src, "--audio-root", "data/audio_s2s", "--out", f"data/tc_en_s2s.{split}.audio.jsonl",
        "--split", sp, "--voices", ",".join(USER_VOICES), "--assistant-voice", ASSIST_VOICE,
        "--concurrency", "8"], check=True)

## 6. Packing INTERLEAVED (le tour final = texte + audio)

In [ ]:
import json, shutil
import s2s_toolcalling.tools.schemas as s
open("tools_en.json", "w").write(json.dumps(s.TOOLCALLING_EN_TOOL_DEFINITIONS))
from s2s_toolcalling.data.preprocess_sft import main as preprocess
for split, out in [("train", "datasets/tc_en_s2s_train"), ("val", "datasets/tc_en_s2s_val")]:
    shutil.rmtree(out, ignore_errors=True)
    preprocess(["--dialogues", f"data/tc_en_s2s.{split}.audio.jsonl", "--audio-root", "data/audio_s2s",
                "--output", out, "--tool-definitions", "tools_en.json",
                "--assistant-audio-mode", "interleaved",
                "--interleaved-text-tokens", "6", "--interleaved-audio-tokens", "12"])

## 7. Entrainer (interleaved ; wandb + push HF)

In [ ]:
from s2s_toolcalling.training.train_sft import TrainConfig, build_trainer
cfg = TrainConfig.from_yaml("configs/phase_b_s2s.yaml")
cfg.train_dataset, cfg.val_dataset, cfg.hub_repo = "datasets/tc_en_s2s_train", "datasets/tc_en_s2s_val", HUB_ADAPTER
n = sum(1 for _ in open("data/tc_en_s2s.train.audio.jsonl"))
cfg.max_steps = max(500, min(cfg.max_steps, 2 * n // cfg.batch_size))
print("max_steps:", cfg.max_steps)
trainer = build_trainer(cfg)
trainer.train()       # surveille text_ppl + val_text_ppl + grad_norm sur wandb

## 8. Demo S2S : audio → tool → reponse PARLEE (+ latence)

In [ ]:
import time, soundfile as sf, torch, numpy as np
from liquid_audio import LFM2AudioModel, LFM2AudioProcessor
from s2s_toolcalling.training.lora import inject_lora, load_lora, load_lora_settings, merge_lora
from s2s_toolcalling.tools.toolcalling_en import build_toolcalling_en_registry
from s2s_toolcalling.orchestrator.agent import ReceptionAgent, AgentConfig
from s2s_toolcalling.orchestrator.events import AudioChunk, ToolCallBegin, ToolCallResult, TextDelta, TurnComplete
from s2s_toolcalling.data.chat_format import TOOLCALLING_EN_SYSTEM_INSTRUCTIONS

BASE = "LiquidAI/LFM2.5-Audio-1.5B"; ADAPTER = "outputs/phase_b_s2s/adapter"
model = LFM2AudioModel.from_pretrained(BASE, device="cuda").eval()
proc = LFM2AudioProcessor.from_pretrained(BASE, device="cuda")
st = load_lora_settings(ADAPTER); inject_lora(model, st)
load_lora(model, ADAPTER + "/adapter_model.safetensors"); merge_lora(model)

from s2s_toolcalling.tools.fake_db import FakeDbBackend
from s2s_toolcalling.tools.web_search import DuckDuckGoBackend
registry = build_toolcalling_en_registry(web_backend=DuckDuckGoBackend(max_results=4), db_backend=FakeDbBackend())
agent = ReceptionAgent(model, proc, registry,      # hybride : tool call texte -> reponse parlee
                       config=AgentConfig(system_instructions=TOOLCALLING_EN_SYSTEM_INSTRUCTIONS))

# une question held-out (Phase A test)
import glob
wavp = sorted(glob.glob("data/tc_en/audio_test/*.wav"))[0]
wav, sr = sf.read(wavp, dtype="float32"); wav = torch.from_numpy(wav)
print("question:", wavp)

chat = agent.new_session()
t0 = time.time(); first_audio = tool_t = None; chunks = []; said = []
for ev in agent.respond(chat, wav, sr):
    if isinstance(ev, ToolCallBegin): print(f"  ⏱ {time.time()-t0:.2f}s tool: {ev.name}({ev.arguments})")
    elif isinstance(ev, ToolCallResult): tool_t = ev.elapsed_ms; print(f"  ⏱ {time.time()-t0:.2f}s result ({ev.elapsed_ms:.0f}ms): {ev.payload}")
    elif isinstance(ev, AudioChunk):
        first_audio = first_audio or (time.time() - t0); chunks.append(ev.samples.numpy().reshape(-1))
    elif isinstance(ev, TextDelta): said.append(ev.text)
    elif isinstance(ev, TurnComplete): print("  texte:", ev.text)
print(f"\n1er son réponse (après round-trip) : {first_audio:.2f}s · outil {tool_t:.0f}ms")
if chunks:
    from IPython.display import Audio, display
    display(Audio(np.concatenate(chunks), rate=24_000))

## 9. Démo WebRTC mains-libres (web_search live + fake DB)
Démo S2S complète, même UX que `s2s_webrtc_demo.py` :

```
!{sys.executable} scripts/s2s_toolcalling_webrtc_demo.py --share --turn cloudflare \
    --checkpoint LiquidAI/LFM2.5-Audio-1.5B --adapter Rcarvalo/lfm25-tc-en-s2s-adapter
```

Tu parles → tool call (texte, propre) → web_search (ddgs) / db_query (fake DB) → réponse
**parlée** ancrée. L'agent est hybride (sequential pour le tool call, interleaved pour parler).

## Suite
- **Latence** : prefix caching via le plugin vLLM-Omni sur le round-trip ; lancer la recherche
  dès le parse de l'argument ; fillers vocaux EN pré-rendus (`FillerBank` + `filler_dir`).
- **db_query réel** : remplacer `FakeDbBackend` par un NL→SQL sur `sql/schema_en.sql`.
